# Insights Lab: Insights Generation

Generate insights from the cleaned dataset and export summary JSON for the app.

In [ ]:
from pathlib import Path

import json
import joblib
import numpy as np
import pandas as pd

ARTIFACTS_DIR = Path('..') / 'artifacts'
df = pd.read_csv(ARTIFACTS_DIR / 'cleaned_dataset.csv')
model = joblib.load(ARTIFACTS_DIR / 'placement_pipeline.joblib')

feature_cols = ['cgpa', 'internships', 'work_experience', 'branch', 'gender']
df['work_experience'] = df.get('work_experience', df['internships']).fillna(df['internships'])
X = df[feature_cols]

proba = model.predict_proba(X)[:, 1]
df['placement_probability'] = proba

In [ ]:
placed = df[df['placed'] == 1]
not_placed = df[df['placed'] == 0]

cgpa_thresholds = {
    'overall_median': round(df['cgpa'].median(), 2),
    'placed_median': round(placed['cgpa'].median(), 2),
    'not_placed_median': round(not_placed['cgpa'].median(), 2)
}

branch_performance = (
    df.groupby('branch')
      .agg(placement_rate=('placed', 'mean'), avg_salary=('salary', 'mean'))
      .reset_index()
)
branch_performance['placement_rate'] = (branch_performance['placement_rate'] * 100).round(1)
branch_performance['avg_salary'] = branch_performance['avg_salary'].round(2)

salary_trends = {
    'overall_avg': round(df['salary'].mean(), 2),
    'placed_avg': round(placed['salary'].mean(), 2),
    'not_placed_avg': round(not_placed['salary'].mean(), 2)
}

baseline_profile = df[feature_cols].median(numeric_only=True).to_dict()
baseline_profile['branch'] = df['branch'].mode().iloc[0]
baseline_profile['gender'] = df['gender'].mode().iloc[0]

baseline_df = pd.DataFrame([baseline_profile])
baseline_prob = float(model.predict_proba(baseline_df)[:, 1][0])

baseline_prediction = {
    'input_profile': baseline_profile,
    'placement_probability': round(baseline_prob * 100, 1)
}

summary = {
    'cgpa_thresholds': cgpa_thresholds,
    'branch_performance': branch_performance.sort_values('placement_rate', ascending=False).to_dict(orient='records'),
    'salary_trends': salary_trends,
    'baseline_prediction': baseline_prediction
}

def to_jsonable(value):
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    return value

summary_path = ARTIFACTS_DIR / 'insights_summary.json'
summary_path.write_text(json.dumps(summary, default=to_jsonable, indent=2), encoding='utf-8')

summary_path